In [ ]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.makedirs("reports/figures", exist_ok=True)
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import umap
from src.training.trainer import load_trained_model
from src.data.dataset import build_tensors
from src.training.health import posterior_collapse
from src.evaluation.latent_analysis import (
    all_latents,
    decoder_weight_matrices,
    load_lineage,
    modality_energy,
    per_sample_mse
)

In [ ]:
model = load_trained_model("outputs/checkpoints/shallow_beta1_seed42.pt")

In [ ]:
tensors, boundary = build_tensors()
X_all = pd.concat([tensors["train"], tensors["val"], tensors["test"]])

In [ ]:
mu, logvar = all_latents(model, X_all, boundary)
assert mu.shape == (904, 128)

In [ ]:
W_tx, W_mt = decoder_weight_matrices(model)
W_tx.shape, W_mt.shape

In [ ]:
lineage = load_lineage("data/raw/Model.csv", mu.index)
print(f"coverage: {lineage.notna().sum()}/{len(lineage)}")
print(lineage.value_counts().head(10))

In [ ]:
n_active, kl_per_dim = posterior_collapse(
    torch.tensor(mu.values),
    torch.tensor(logvar.values),
)
print(f"active {n_active}/128")

In [ ]:
kl = kl_per_dim.numpy()
plt.bar(range(128), sorted(kl, reverse=True))
plt.xlabel("latent dimension (sorted)"); plt.ylabel("KL")
plt.savefig("reports/figures/_active_dims.png", dpi=150, bbox_inches="tight")

In [ ]:
tx_e, mt_e = modality_energy(W_tx, W_mt)
tx_n = tx_e / tx_e.max()
mt_n = mt_e / mt_e.max()
both = ((tx_n > 0.1) & (mt_n > 0.1)).mean()
print(f"dims active on both sides: {both:.0%}")

All 128 dimensions stay active at beta = 1, the model spread its work across the whole space rather than leaning on a few dimensions. No collapse

In [ ]:
for t in (0.05, 0.10, 0.20):
    b = ((tx_n > t) & (mt_n > t)).mean()
    print(f"{t}: {b:.0%}")

In [ ]:
plt.scatter(tx_e, mt_e)
plt.xlabel("gene-side push (per-feature)"); plt.ylabel("metabolite-side push (per-feature)")
plt.savefig("reports/figures/cross_modality_coherence.png", dpi=150, bbox_inches="tight")

The two data types barely correlate in the raw numbers (r≈0.045), so checking to see if the model fused them or stacked them. The shared space is genuinely joint, not two different spaces bolted together.

In [ ]:
from scipy.stats import spearmanr
rho, _ = spearmanr(tx_e, mt_e)
print(f"rank corr: {rho:+.3f}")

In [ ]:
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
emb2d = reducer.fit_transform(mu.values)

codes, labels = pd.factorize(lineage)
plt.figure(figsize=(9, 7))
sc = plt.scatter(emb2d[:, 0], emb2d[:, 1], c=codes, cmap="tab20", s=12)
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
plt.savefig("reports/figures/sample_umap.png", dpi=150, bbox_inches="tight")

904 cell lines, latents squashed to 2D, colored by cancer lineage (not seen in training). Several lineages form distinct islands, carcinoma type lineages sit closer together on the right, which is expected since they share biology. The model organised cell lines by biology without being given the labels

In [ ]:
mse_tx, mse_mt = per_sample_mse(model, X_all, boundary)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(mse_tx, bins=40); ax[0].set_title("gene MSE per sample")
ax[1].hist(mse_mt, bins=40); ax[1].set_title("metabolite MSE per sample")
plt.savefig("reports/figures/reconstruction_mse.png", dpi=150, bbox_inches="tight")

Per-sample rebuild error, genes and metabolites separately. Both single-peaked, no cluster of badly-rebuilt lines. Metabolites rebuild tighter than genes, consistent with mt R² 0.60 > tx 0.44.